In [1]:
import numpy as np
print(np.__version__)

2.2.6


In [2]:
import sys
import os
sys.path.append(os.path.abspath('../'))

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch

from tqdm import tqdm

project_root = Path().resolve().parent
sys.path.append(str(project_root))

from src.validation import get_tpcf

### Helpers and Functions

In [3]:
#### Calculate 2pcf for sampled
def calc_tpcf_gen(gen_array, scale=True):
    """
    gen array must have size: # cosmology, # repeats, # particles, 3
    """
    gen_tpcf = []
    for i in tqdm(range(gen_array.shape[0])):
        inter_gen_tpcf = []
        for j in range(gen_array[i].shape[0]):
            if scale:
                inter_gen_tpcf.append(get_tpcf(gen_array[i][j] * 1000, boxsize, r_bins, mu_bins))
            else:
                inter_gen_tpcf.append(get_tpcf(gen_array[i][j], boxsize, r_bins, mu_bins))
        gen_tpcf.append(np.array(inter_gen_tpcf))
    gen_tpcf = np.array(gen_tpcf)

    return gen_tpcf

In [4]:
#### Calculate 2pcf for simulated
def calc_tpcf_true(true_array, scale=True):
    sim_tpcf = []
    for i in tqdm(range(true_array.shape[0])):
        if scale:
            sim_tpcf.append(get_tpcf(true_array[i] * 1000, boxsize, r_bins, mu_bins))
        else:
            sim_tpcf.append(get_tpcf(true_array[i], boxsize, r_bins, mu_bins))
    sim_tpcf = np.array(sim_tpcf)

    return sim_tpcf

In [5]:
#### Calculate MAE
def calc_mae(sim_tpcf, gen_tpcf):
    
    mae = np.mean(np.absolute(sim_tpcf[:, None] - gen_tpcf), axis=(0, 1))
    mae_total = np.mean(mae)
    
    print(f'''
    MAE:", {mae}
    
    AVG MAE:", {mae_total}''')

    return mae, mae_total

In [6]:
#### Calculate MSE 
def calc_mse(sim_tpcf, gen_tpcf):
    mse = np.mean((sim_tpcf[:, None] - gen_tpcf)**2, axis=(0, 1))
    mse_total = np.mean(mse)
    print(f'''
    MSE:", {mse}
    
    AVG MSE:", {mse_total}''')

    return mse, mse_total

### Data Loading and Preprocessing

In [7]:
# Two-Point Correlation function
boxsize = 1000.
r_bins = np.linspace(0.5, 150.0, 25) # Defines 25 linearly spaced bins from 0.5 to 150 for the radial separation r
r_c = 0.5*(r_bins[1:] + r_bins[:-1]) # Computes the center of each radial bin (midpoint between two consecutive bins)
mu_bins = np.linspace(-1, 1, 201)

In [8]:
# Generated samples
train_run = "20260312_121618"
infer_run = "20260312_134739"
gen_samples = torch.load(f"/gpfs/home4/bartb/T5000/results/run_{train_run}/quantitative_2pcf/{infer_run}/gen_samples.pth", weights_only=False)
print(f"Shape of generated samples: {gen_samples.shape}")

batches, batch_size, repeats, halos, D = gen_samples.shape[0], gen_samples.shape[1], gen_samples.shape[2], gen_samples.shape[3], gen_samples.shape[4]
gen_samples = gen_samples.reshape(batches*batch_size, repeats, halos, D)
params = torch.load(f"/gpfs/home4/bartb/T5000/results/run_{train_run}/quantitative_2pcf/{infer_run}/cond.pth", weights_only=False)
print(f"Shape of generated samples: {gen_samples.shape}")
print(f"Shape of parameter tensor: {params.shape}")
# print(gen_samples[0][0])

FileNotFoundError: [Errno 2] No such file or directory: '/gpfs/home4/bartb/T5000/results/run_20260312_121618/quantitative_2pcf/20260312_134739/gen_samples.pth'

In [ ]:
# True validation samples
true_samples = torch.from_numpy(np.load("/gpfs/home4/bartb/T5000/Data/test_halos.npy")[..., :3])
print(f"Shape of true samples: {true_samples.shape}")
true_2pcf_quant = calc_tpcf_true(true_samples, False)

In [ ]:
gen_2pcf_quant = calc_tpcf_gen(gen_samples, True)

In [ ]:
# Calculate MSE and MAE
mae, mae_total = calc_mae(true_2pcf_quant, gen_2pcf_quant)
mse, mse_total = calc_mse(true_2pcf_quant, gen_2pcf_quant)

# print(f'''
# MAE: {mae}
# MSE: {mse}
# Avg MAE: {mae_total}
# Avg MSE: {mse_total}
# ''')

In [ ]:
# Visualization of a generated sample
halos = gen_samples[50][0]
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")


ax.scatter(halos[:, 0], halos[:, 1], halos[:, 2], s=1)


ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")


plt.show()


In [ ]:
# Visualization of a true sample
halos = true_samples[50]
print(halos.shape)
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection="3d")


ax.scatter(halos[:, 0], halos[:, 1], halos[:, 2], s=1)


ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")


plt.show()


In [ ]:
sample = torch.rand(5, 5).repeat_interleave(20, dim=0)
batch = torch.arange(0, 100).repeat_interleave(5000, dim=0)
sample[batch]